In [1]:
from PIL import (
    Image,
    ImageEnhance,
    ImageFilter,
    ImageChops
)

import os
import io

# =====================
# 設定
# =====================

INPUT_DIR = r"xxxx
OUTPUT_DIR = r"xxxx
OUTPUT_SIZE = 1000

MAX_SIZE = 499 * 1024

BG_COLOR = (255, 255, 255)

TARGET_OCCUPANCY = 0.90
MAX_OCCUPANCY = 0.94

# =====================
# 高品質設定
# =====================

# 内部拡大倍率
UPSCALE = 2

# 彩度
SATURATION = 1.06

# シャープ
SHARPNESS = 1.18

# コントラスト
CONTRAST = 1.04

# 明るさ
BRIGHTNESS = 1.01

# ノイズ軽減
SMOOTH = 0.25

# =====================
# フォルダ作成
# =====================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================
# 余白削除
# =====================

def trim_whitespace(
    img,
    bg_color=(255,255,255)
):

    bg = Image.new(
        img.mode,
        img.size,
        bg_color
    )

    diff = ImageChops.difference(
        img,
        bg
    )

    bbox = diff.getbbox()

    if bbox:
        return img.crop(bbox)

    return img

# =====================
# 一括処理
# =====================

for fname in os.listdir(INPUT_DIR):

    if not fname.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    ):
        continue

    print(f"\n処理中: {fname}")

    in_path = os.path.join(
        INPUT_DIR,
        fname
    )

    base_name = os.path.splitext(fname)[0]

    out_path = os.path.join(
        OUTPUT_DIR,
        base_name + ".jpg"
    )

    # =====================
    # 読み込み
    # =====================

    img = Image.open(in_path)

    # PNG透過対応
    if img.mode in ("RGBA", "LA"):

        bg = Image.new(
            "RGB",
            img.size,
            (255,255,255)
        )

        bg.paste(
            img,
            mask=img.split()[-1]
        )

        img = bg

    else:
        img = img.convert("RGB")

    # =====================
    # 余白削除
    # =====================

    img = trim_whitespace(img)

    # =====================
    # CPU高品質アップスケール
    # =====================

    w, h = img.size

    # 段階拡大（重要）
    img = img.resize(
        (
            int(w * 1.5),
            int(h * 1.5)
        ),
        Image.LANCZOS
    )

    w2, h2 = img.size

    img = img.resize(
        (
            int(w2 * 1.3),
            int(h2 * 1.3)
        ),
        Image.LANCZOS
    )

    # =====================
    # ノイズ軽減
    # =====================

    img = img.filter(
        ImageFilter.SMOOTH_MORE
    )

    # =====================
    # シャープ
    # =====================

    sharp = ImageEnhance.Sharpness(img)

    img = sharp.enhance(
        SHARPNESS
    )

    # =====================
    # コントラスト
    # =====================

    contrast = ImageEnhance.Contrast(img)

    img = contrast.enhance(
        CONTRAST
    )

    # =====================
    # 明るさ
    # =====================

    bright = ImageEnhance.Brightness(img)

    img = bright.enhance(
        BRIGHTNESS
    )

    # =====================
    # 彩度
    # =====================

    color = ImageEnhance.Color(img)

    img = color.enhance(
        SATURATION
    )

    # =====================
    # アンシャープ
    # =====================

    img = img.filter(

        ImageFilter.UnsharpMask(
            radius=1.6,
            percent=140,
            threshold=2
        )

    )

    # =====================
    # サイズ調整
    # =====================

    w, h = img.size

    scale = (
        OUTPUT_SIZE * MAX_OCCUPANCY
    ) / max(w, h)

    min_scale = (
        OUTPUT_SIZE * TARGET_OCCUPANCY
    ) / max(w, h)

    scale = max(scale, min_scale)

    new_w = int(w * scale)
    new_h = int(h * scale)

    img = img.resize(
        (new_w, new_h),
        Image.LANCZOS
    )

    # =====================
    # 正方形化
    # =====================

    canvas = Image.new(
        "RGB",
        (OUTPUT_SIZE, OUTPUT_SIZE),
        BG_COLOR
    )

    x = (
        OUTPUT_SIZE - new_w
    ) // 2

    y = (
        OUTPUT_SIZE - new_h
    ) // 2

    canvas.paste(
        img,
        (x, y)
    )

    # =====================
    # 容量調整
    # =====================

    quality = 99

    while quality >= 30:

        buffer = io.BytesIO()

        canvas.save(
            buffer,
            format="JPEG",
            quality=quality,
            optimize=True
        )

        size = buffer.tell()

        if size <= MAX_SIZE:

            with open(out_path, "wb") as f:
                f.write(buffer.getvalue())

            print(
                f"✔ 完了: "
                f"{base_name}.jpg "
                f"({round(size/1024)}KB)"
            )

            break

        quality -= 5

print("\n🎉 全画像処理完了")


処理中: 4570100013201.png
✔ 完了: 4570100013201.jpg (245KB)

処理中: 4570100013201_1.jpg
✔ 完了: 4570100013201_1.jpg (216KB)

処理中: 4570100013201_2.png
✔ 完了: 4570100013201_2.jpg (221KB)

🎉 全画像処理完了
